# 03 · De la serie semanal a la ABT

Cada fila representa un repuesto en una semana **recién cerrada (t)**.
Las features usan t y el pasado; los targets contienen las ventas de las cuatro
semanas siguientes. No entrenamos ni seleccionamos repuestos en este notebook.

Usamos datos inventados para poder verificar los números a mano. Ejecutá las
celdas en orden desde esta carpeta, con el entorno `stockifai`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../src")

import pandas as pd
from stockifai_forecast.features import construir_abt

## 1. Dos repuestos, dieciséis semanas

Las dos series comienzan con una venta y tienen semanas consecutivas con ceros,
como la salida de `rellenar_ceros`. Los repuestos tienen magnitudes distintas
para que sea fácil detectar si se mezclan.

In [3]:
semanas = pd.date_range("2025-06-02", periods=16, freq="W-MON")
serie = pd.DataFrame({
    "repuesto": ["A"] * 16 + ["B"] * 16,
    "semana": list(semanas) * 2,
    "cantidad": [2, 0, 4, 0, 0, 6, 0, 0, 0, 3, 0, 0, 5, 0, 0, 1]
    + [10, 0, 0, 20, 0, 0, 0, 30, 0, 0, 0, 0, 40, 0, 0, 0],
})
serie

,repuesto,semana,cantidad
0,A,2025-06-02,2
1,A,2025-06-09,0
2,A,2025-06-16,4
3,A,2025-06-23,0
4,A,2025-06-30,0
5,A,2025-07-07,6
6,A,2025-07-14,0
7,A,2025-07-21,0
8,A,2025-07-28,0
9,A,2025-08-04,3


## 2. La ABT completa

No se elimina ninguna fila. Los `NaN` iniciales indican historia insuficiente;
los finales, ventas futuras todavía desconocidas. Son situaciones distintas.
Al entrenar habrá que elegir las filas con target conocido para cada horizonte;
en inferencia conservamos las últimas filas con targets desconocidos.

In [4]:
abt = construir_abt(serie)
pd.set_option("display.max_columns", None)
abt

,repuesto,semana,cantidad,lag_1,lag_2,lag_3,prom_4,prom_8,prom_12,desvio_8,semanas_con_venta_8,semanas_desde_ultima_venta,mes,semana_del_anio,feriados_t1,feriados_t2,feriados_t3,feriados_t4,target_1,target_2,target_3,target_4
0,A,2025-06-02,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,6,23,0,2,0,0,0.0,4.0,0.0,0.0
1,A,2025-06-09,0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,6,24,2,0,0,1,4.0,0.0,0.0,6.0
2,A,2025-06-16,4,0.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,6,25,0,0,1,0,0.0,0.0,6.0,0.0
3,A,2025-06-23,0,4.0,0.0,2.0,1.50,NaN,NaN,NaN,NaN,1.0,6,26,0,1,0,0,0.0,6.0,0.0,0.0
4,A,2025-06-30,0,0.0,4.0,0.0,1.00,NaN,NaN,NaN,NaN,2.0,6,27,1,0,0,0,6.0,0.0,0.0,0.0
5,A,2025-07-07,6,0.0,0.0,4.0,2.50,NaN,NaN,NaN,NaN,0.0,7,28,0,0,0,0,0.0,0.0,0.0,3.0
6,A,2025-07-14,0,6.0,0.0,0.0,1.50,NaN,NaN,NaN,NaN,1.0,7,29,0,0,0,2,0.0,0.0,3.0,0.0
7,A,2025-07-21,0,0.0,6.0,0.0,1.50,1.500,NaN,2.329929,3.0,2.0,7,30,0,0,2,0,0.0,3.0,0.0,0.0
8,A,2025-07-28,0,0.0,0.0,6.0,1.50,1.250,NaN,2.375470,2.0,3.0,7,31,0,2,0,0,3.0,0.0,0.0,5.0
9,A,2025-08-04,3,0.0,0.0,0.0,0.75,1.625,NaN,2.386719,3.0,0.0,8,32,2,0,0,0,0.0,0.0,5.0,0.0


## 3. Revisar números a mano

En la octava semana de A, las últimas cuatro cantidades son `[0, 6, 0, 0]`:
`prom_4 = 1,5`. En las ocho semanas hubo tres con venta, y la última venta fue
dos semanas antes. El desvío muestral es `sqrt(38 / 7) ≈ 2,33`.
La ventana incluye la cantidad de t, porque esa semana ya cerró.

In [5]:
columnas = ["repuesto", "semana", "cantidad", "lag_1", "prom_4",
            "desvio_8", "semanas_con_venta_8", "semanas_desde_ultima_venta",
            "target_1", "target_2", "target_3", "target_4"]
abt.loc[(abt["repuesto"] == "A") & (abt["semana"] == semanas[7]), columnas]

,repuesto,semana,cantidad,lag_1,prom_4,desvio_8,semanas_con_venta_8,semanas_desde_ultima_venta,target_1,target_2,target_3,target_4
7,A,2025-07-21,0,0.0,1.5,2.329929,3.0,2.0,0.0,3.0,0.0,0.0


## 4. La frontera entre repuestos

La última fila de A no tiene `target_1`, aunque debajo esté B.
La primera fila de B no tiene `lag_1` ni `prom_4`: su historia empieza ahí.

In [6]:
frontera = pd.concat([
    abt[abt["repuesto"] == "A"].tail(2),
    abt[abt["repuesto"] == "B"].head(2),
])
frontera[columnas]

,repuesto,semana,cantidad,lag_1,prom_4,desvio_8,semanas_con_venta_8,semanas_desde_ultima_venta,target_1,target_2,target_3,target_4
14,A,2025-09-08,0,0.0,1.25,1.927248,2.0,2.0,1.0,NaN,NaN,NaN
15,A,2025-09-15,1,0.0,1.50,1.885092,3.0,0.0,NaN,NaN,NaN,NaN
16,B,2025-06-02,10,NaN,NaN,NaN,NaN,0.0,0.0,0.0,20.0,0.0
17,B,2025-06-09,0,10.0,NaN,NaN,NaN,1.0,0.0,20.0,0.0,0.0


## 5. Qué sale mal sin `groupby`

`shift(1)` toma la fila anterior de toda la tabla. En la primera fila de B
trae la última cantidad de A (1), aunque sea otro repuesto y otra fecha.
`groupby("repuesto")["cantidad"].shift(1)` reinicia el historial en cada SKU.
Para los targets pasa lo mismo, pero mirando hacia adelante con `shift(-1)`.

In [7]:
comparacion = serie.sort_values(["repuesto", "semana"]).reset_index(drop=True)
comparacion["lag_sin_groupby_ERROR"] = comparacion["cantidad"].shift(1)
comparacion["lag_por_repuesto_OK"] = comparacion.groupby("repuesto")["cantidad"].shift(1)
comparacion["target_sin_groupby_ERROR"] = comparacion["cantidad"].shift(-1)
comparacion["target_por_repuesto_OK"] = comparacion.groupby("repuesto")["cantidad"].shift(-1)
comparacion.iloc[14:18]

,repuesto,semana,cantidad,lag_sin_groupby_ERROR,lag_por_repuesto_OK,target_sin_groupby_ERROR,target_por_repuesto_OK
14,A,2025-09-08,0,0.0,0.0,1.0,1.0
15,A,2025-09-15,1,0.0,0.0,10.0,NaN
16,B,2025-06-02,10,1.0,NaN,0.0,0.0
17,B,2025-06-09,0,10.0,10.0,0.0,0.0


## 6. Calendario futuro conocido

En la fila del 30 de junio, `feriados_t1` cuenta los feriados de la semana que
empieza el 7 de julio: hay uno, el 9 de julio. Se cuentan días nacionales de
lunes a domingo, sin feriados provinciales. El calendario puede usarse aunque
todavía no conozcamos las ventas de esas semanas.

In [8]:
abt.loc[abt["semana"] == pd.Timestamp("2025-06-30"),
        ["repuesto", "semana", "feriados_t1", "feriados_t2", "feriados_t3", "feriados_t4"]]

,repuesto,semana,feriados_t1,feriados_t2,feriados_t3,feriados_t4
4,A,2025-06-30,1,0,0,0
20,B,2025-06-30,1,0,0,0


## 7. Comprobar que no miramos ventas futuras

Cambiamos a 999 las cantidades posteriores al corte. Las features de la fila
de corte deben quedar idénticas; los targets sí deben cambiar.

In [9]:
corte = pd.Timestamp("2025-07-21")
otra_serie = serie.copy()
otra_serie.loc[otra_serie["semana"] > corte, "cantidad"] = 999
otra_abt = construir_abt(otra_serie)
features = [c for c in abt.columns if not c.startswith("target_")]
filas = abt["semana"] == corte
pd.testing.assert_frame_equal(abt.loc[filas, features], otra_abt.loc[filas, features])
print("OK: cambiar las ventas futuras no cambió las features del corte.")
pd.concat([
    abt.loc[filas, ["repuesto", "semana", "target_1"]].assign(escenario="original"),
    otra_abt.loc[filas, ["repuesto", "semana", "target_1"]].assign(escenario="futuro modificado"),
])

OK: cambiar las ventas futuras no cambió las features del corte.


,repuesto,semana,target_1,escenario
7,A,2025-07-21,0.0,original
23,B,2025-07-21,0.0,original
7,A,2025-07-21,999.0,futuro modificado
23,B,2025-07-21,999.0,futuro modificado


## Cómo leer el código de `construir_abt`, en orden

1. Valida parámetros, copia las tres columnas y ordena por repuesto y semana.
2. Agrupa por repuesto y calcula lags, promedios y desvío, sin mezclar historias.
3. Cuenta cuántas de las últimas ocho semanas tuvieron venta.
4. Conserva la fecha de la última venta conocida y calcula su distancia a t.
5. Agrega mes y número de semana ISO de t.
6. Cuenta los feriados nacionales de cada semana futura conocida por calendario.
7. Agrega las cantidades futuras como targets, sin completar ni eliminar NaN.

### Métodos de pandas, con ejemplos

| Método | Ejemplo o significado |
|---|---|
| `copy()` | Trabaja sobre otra tabla y deja intacta la original. |
| `sort_values()` y `reset_index(drop=True)` | Ordenan filas cronológicamente y renumeran el índice. |
| `groupby("repuesto")` | Separa las cuentas por SKU, parecido a `PARTITION BY` en SQL. |
| `shift(1)` | `[2, 0, 4]` pasa a `[NaN, 2, 0]`: trae el pasado a la fila actual. |
| `shift(-1)` | `[2, 0, 4]` pasa a `[0, 4, NaN]`: trae el futuro como etiqueta. |
| `rolling(4, min_periods=4).mean()` | Como arrastrar en Excel un promedio de cuatro celdas: solo produce valor con cuatro semanas. |
| `transform(...)` | Calcula dentro de cada repuesto y devuelve un valor alineado con cada fila original; no resume a una sola fila por SKU. |
| `std(ddof=1)` | Desvío muestral: usa n−1 en el denominador. Es la convención elegida para `desvio_8`. |
| `gt(0)` | Compara con cero: `[2, 0, 4]` da `[True, False, True]`; sumarlos cuenta dos semanas con venta. |
| `where(...)` | Conserva la fecha solo cuando hubo venta; en el resto pone `NaT` (fecha ausente). |
| `ffill()` por grupo | `[fecha1, NaT, fecha3]` queda `[fecha1, fecha1, fecha3]`: arrastra la última fecha conocida. Nunca busca una venta futura. |
| `Timedelta(weeks=h)` | Desplaza una fecha h semanas. Restar fechas y dividir días por siete mide semanas transcurridas. |
| `.dt.month` / `.dt.isocalendar().week` | Extraen mes y semana ISO. El 29/12/2025 pertenece a la semana ISO 1 de 2026. |
| `drop_duplicates()` y `map(...)` | Calculan el calendario una vez por semana distinta y asignan su conteo a todas las filas de esa semana. |

### Decisiones adicionales

- Entrada esperada: salida de `rellenar_ceros`, con lunes consecutivos y claves únicas.
  Esta función no deduplica ni repara la serie.
- Antes de la primera venta, si recibiera ceros iniciales, la antigüedad queda
  `NaN`: todavía no existe una última venta conocida.
- Se reinicia el índice tras ordenar, sin conservar el índice original como feature.
- Se usa el calendario nacional predeterminado de `holidays.Argentina`, sin
  subdivisiones. Los años cubren todo el horizonte, incluso al cruzar diciembre.
- Se aceptan otras ventanas y otro horizonte; las dos features de ocho semanas
  siguen usando ocho semanas, como indica su nombre.
- No se agrega segmento ni elegibilidad: se calcularán después usando train.
- `holidays` ya estaba declarado en `requirements.txt`; no hizo falta modificarlo.

### Qué verifican los tests

| Test | Qué protege |
|---|---|
| Frontera | Lags, ventanas y targets no pasan de un repuesto a otro. |
| Anti-leakage | Cambiar ventas posteriores al corte no cambia ninguna feature del corte. |
| Targets | Cada etiqueta coincide con la venta del mismo SKU h semanas después. |
| Valores a mano | Promedios, lags, cantidad actual, desvío, frecuencia y distancia tienen los números esperados. |
| NaN | Se conserva la falta de historia y de futuro sin inventar valores. |
| Filas e input | No se pierden claves, no aparecen duplicados y no se modifica la entrada. |
| Orden | Mezclar las filas de entrada da la misma ABT. |
| 9 de julio | El feriado cae en la semana y horizonte correctos. |
| Cambio de año | El calendario futuro y las semanas ISO cruzan diciembre correctamente. |
| Parámetros | Se respetan ventanas y horizontes diferentes de los predeterminados. |
| Sin venta previa | No se inventa una última venta ni se hereda de otro SKU. |
| Tabla vacía | Se devuelve una ABT vacía con las columnas esperadas. |
| Parámetros inválidos | Ventanas y horizonte no positivos producen un error claro. |

Los nueve tests anteriores de `datos.py` también deben seguir pasando.
